# Your First Machine Learning Model: K-Nearest Neighbours (KNN)

Welcome! In the next 30 minutes you will train a real machine learning model — no maths degree needed.

**The data:** every *bulk* and *block* share deal on the Indian stock market (almost all NSE, a handful from BSE) from 2020 to today — about 1.2 lakh real deals.

**The mission:** teach the computer to look at a deal — how many shares, at what price, worth how many crores — and guess whether it was a **BULK** deal or a **BLOCK** deal.

**How to use this notebook:** click a grey code cell, then press the ▶ button on its left (or press `Shift + Enter`). Run the cells **in order, top to bottom** — later cells use things created by earlier ones. Text cells like this one are just for reading.

**Three things worth knowing before you start:**

1. The **first time you press ▶**, Colab shows a warning that this notebook *"was not authored by Google"*. That's normal for any notebook loaded from GitHub — click **Run anyway**.
2. The first cell takes a few extra seconds while Colab connects to a computer in the cloud. Not broken, just warming up.
3. If you ever see an error like `NameError: name 'deals' is not defined`, or Colab says the runtime disconnected, don't panic: go to the menu and click **Runtime → Run all**. Everything comes back in under a minute.


## How KNN works — the one-minute version

Imagine you move to a new city and want to know if a house is in a "posh" area or a "normal" area. What do you do? You look at the **5 nearest houses**. If 4 of them are posh, you say: posh area.

That is the entire idea of K-Nearest Neighbours:

1. To label something new, find the **K most similar** examples you have already seen (K = 5 nearest houses).
2. Let those neighbours **vote**.
3. The majority wins.

"Most similar" simply means: closest in numbers. Two deals with almost the same quantity, price and value are neighbours. That's it — KNN never really "learns rules", it just remembers every example and compares.

---

### A quick word on BULK vs BLOCK deals

* A **BULK deal** is when someone buys or sells more than 0.5% of a company's shares over a normal trading day.
* A **BLOCK deal** is one single, giant, pre-negotiated trade (crores of rupees in one shot) done in a special trading window.

Block deals tend to be much bigger in value — that's the pattern we hope the model discovers on its own.


## Step 1 — Load the toolbox

Python has free, ready-made tools for machine learning. We just import them.


In [ ]:
# Step 1: Load our toolbox
import pandas as pd                                     # works with tables of data (like Excel)
from sklearn.model_selection import train_test_split    # splits data into train / test
from sklearn.preprocessing import StandardScaler        # puts all numbers on the same scale
from sklearn.neighbors import KNeighborsClassifier      # the KNN model itself
from sklearn.metrics import (accuracy_score, classification_report,
                             ConfusionMatrixDisplay)    # report cards for the model
import matplotlib.pyplot as plt                         # draws charts

print("Toolbox ready!")

## Step 2 — Load the dataset

The deals data lives on GitHub. One line downloads it straight into a table called `deals`.


In [ ]:
# Step 2: Load the dataset (real bulk & block deals, 2020 to today)
url = "https://raw.githubusercontent.com/Sreenivas-Sadhu-Prabhakara/learn-ml-models/main/data/india_bulk_block_deals.csv"
deals = pd.read_csv(url)

print("Number of deals:", len(deals))
deals.head()   # shows the first 5 rows

Each **row** is one real deal. The columns we care about:

| Column | Meaning |
|---|---|
| `deal_type` | BULK or BLOCK — **this is what we want to predict** |
| `quantity` | how many shares changed hands |
| `price_inr` | price per share in ₹ |
| `trade_value_crore` | total deal value in ₹ crore |


## Step 3 — Get to know the data

Rule #1 of machine learning: look at your data before modelling it.


In [ ]:
# Step 3: How many deals of each type do we have?
deals["deal_type"].value_counts()

**Problem spotted!** About 92% of deals are BULK and only 8% are BLOCK.

Why is that a problem? A lazy model that just shouts "BULK!" every single time would be right 92% of the time — sounds impressive, learns nothing. This is called an **imbalanced dataset**, and it's one of the most common traps in machine learning.


## Step 4 — Make it a fair 50/50 contest

The simplest fix: keep **all** the BLOCK deals, and randomly pick an **equal number** of BULK deals. Now guessing blindly only gets you 50% — the model has to actually learn something to beat that.


In [ ]:
# Step 4: Balance the dataset — equal number of BULK and BLOCK deals
block_deals = deals[deals["deal_type"] == "BLOCK"]
bulk_deals  = deals[deals["deal_type"] == "BULK"].sample(n=len(block_deals), random_state=42)

data = pd.concat([block_deals, bulk_deals])
data["deal_type"].value_counts()

*(`random_state=42` just means "pick randomly, but pick the **same** random deals every time", so everyone in the room gets identical results. The number 42 is a programmer in-joke — any fixed number works.)*


## Step 5 — Choose the clues (X) and the answer (y)

In ML language:

* **Features (X)** = the clues the model may look at → quantity, price, total value
* **Label (y)** = the answer we want it to guess → BULK or BLOCK


In [ ]:
# Step 5: Choose the clues (features) and the answer (label)
features = ["quantity", "price_inr", "trade_value_crore"]

X = data[features]       # the clues
y = data["deal_type"]    # the answer

X.head()

## Step 6 — Split the data: 75% training, 25% testing

This is the **most important honesty rule** in machine learning:

* **75% training set** → the model studies these deals (answers included).
* **25% test set** → kept locked away. The model never sees these during training. This is the final exam.

If you test the model on deals it already studied, it's like grading a student on questions they memorised the night before — the mark means nothing.


In [ ]:
# Step 6: Split — 75% for learning, 25% hidden away for the exam
X_train, X_test, y_train, y_test = train_test_split(
    X, y,
    train_size=0.75,     # 75% training, 25% testing
    random_state=42,     # same split for everyone
    stratify=y,          # keep the 50/50 mix in both halves
)

print("Deals for training:", len(X_train))
print("Deals for testing: ", len(X_test))

## Step 7 — Put all clues on the same scale

KNN works by measuring **distance** between deals. But look at our clues:

* `quantity` is in lakhs (e.g. 7,50,000)
* `price_inr` is in hundreds (e.g. 537)
* `trade_value_crore` is in tens (e.g. 40)

Without fixing this, the giant `quantity` numbers **bully** the distance calculation and the other clues barely matter. `StandardScaler` converts every column so they all speak with the same volume.

One subtle but important detail: we **learn** the scaling from the *training* data only (`fit_transform`), then **apply** it to the test data (`transform`). The exam paper must never influence the study notes.


In [ ]:
# Step 7: Scale the features so no single clue dominates
scaler = StandardScaler()
X_train_scaled = scaler.fit_transform(X_train)   # learn the scale from training data
X_test_scaled  = scaler.transform(X_test)        # apply that same scale to test data

print("All clues are now on the same scale.")

## Step 8 — Train the model

`n_neighbors=5` is the **K** in KNN: to judge a new deal, look at its 5 most similar deals and take a vote. Training a KNN is instant — it simply memorises the training deals.


In [ ]:
# Step 8: Train the KNN model (K = 5 neighbours)
knn = KNeighborsClassifier(n_neighbors=5)
knn.fit(X_train_scaled, y_train)

print("Model trained on", len(X_train), "deals.")

## Step 9 — Exam time!

The model now predicts the deal type for all the test deals — the 25% it has **never seen**. Then we check its answers against the truth.


In [ ]:
# Step 9: Predict the hidden 25% and check the score
predictions = knn.predict(X_test_scaled)

accuracy = accuracy_score(y_test, predictions)
print(f"Accuracy: {accuracy:.1%}")

### Is that good? How to judge the score

Always compare against the **dumbest possible strategy**:

| Strategy | Accuracy |
|---|---|
| Toss a coin | ~50% |
| Our KNN model | **~80%** |

The model gets 4 out of 5 deals right using only three numbers per deal. It genuinely learned the pattern that block deals tend to be bigger-value trades. (Rough guide for a balanced 50/50 problem: ~50% = learned nothing, 60–75% = learned something, 75–90% = solid, 95%+ = check for cheating — usually a leak of the answer into the clues!)


## Step 10 — Judge the model properly

Accuracy is one number. These two tools show **where** the model goes right and wrong.

### 10a. The confusion matrix

A 2×2 scoreboard: rows are the **truth**, columns are the model's **guess**. The diagonal (top-left, bottom-right) is where the model was correct — you want big numbers there.


In [ ]:
# Step 10a: The confusion matrix — where is it right, where does it slip?
ConfusionMatrixDisplay.from_predictions(y_test, predictions, cmap="Greens")
plt.show()

In [ ]:
# Step 10b: The full report card
print(classification_report(y_test, predictions))

### How to read the report card

* **Precision** — when the model *says* BLOCK, how often is it actually BLOCK? (How trustworthy are its claims?)
* **Recall** — of all the *real* BLOCK deals, how many did the model catch? (How much does it miss?)
* **F1-score** — a single number balancing precision and recall.
* **Support** — how many test deals of that type existed.

In real life you care about different ones: a spam filter needs high *precision* (never bin a real email), a cancer screening test needs high *recall* (never miss a real case).


### 10c. Proof that Step 7 (scaling) actually mattered

Let's train the exact same model on the **raw, unscaled** numbers and compare.


In [ ]:
# Step 10c: Same model, but skipping the scaling step
knn_unscaled = KNeighborsClassifier(n_neighbors=5)
knn_unscaled.fit(X_train, y_train)                # raw numbers, no scaling
unscaled_accuracy = accuracy_score(y_test, knn_unscaled.predict(X_test))

print(f"WITHOUT scaling: {unscaled_accuracy:.1%}")
print(f"WITH scaling:    {accuracy:.1%}")

Several percentage points of accuracy — recovered by one line of preprocessing. This is a big lesson of practical ML: **preparing the data well matters as much as the model itself.**


## Step 11 — Tune K: how many neighbours should vote?

K is a dial we chose (we picked 5). Small K = jumpy decisions based on very few neighbours. Huge K = averages over so many neighbours it blurs the pattern. Let's try a range and see.


In [ ]:
# Step 11: Try different values of K and plot the results
k_values = [1, 3, 5, 7, 9, 11, 15, 21, 31, 51]
scores = []

for k in k_values:
    model = KNeighborsClassifier(n_neighbors=k)
    model.fit(X_train_scaled, y_train)
    score = accuracy_score(y_test, model.predict(X_test_scaled))
    scores.append(score)
    print(f"K = {k:>2}  →  accuracy {score:.1%}")

plt.plot(k_values, scores, marker="o")
plt.xlabel("K (number of neighbours who vote)")
plt.ylabel("Accuracy on the test set")
plt.title("Choosing K")
plt.grid(True, alpha=0.3)
plt.show()

**Careful with K = 1!** It often scores high here, but with only one neighbour voting, a single odd or duplicated deal can decide the answer — it's the model equivalent of asking exactly one person for directions. (Our dataset even contains near-twin rows — the buy side and sell side of the same trade — which flatters K = 1.) Odd, modest values like 5–15 are the safe, standard choice; odd numbers also mean the vote can never tie.

**One honest confession:** to keep this lesson short, we compared K values using the test set. In a real project that's cheating — you'd pick K using a separate slice of the *training* data (called a *validation set*), and touch the test set exactly once, at the very end.


## Step 12 — The fun part: invent a deal and ask the model

Make up a deal and see what the model thinks. Change the three numbers and run the cell again!


In [ ]:
# Step 12: Describe an imaginary deal and get the model's verdict
my_deal = pd.DataFrame({
    "quantity":          [2_000_000],    # 20 lakh shares
    "price_inr":         [950.0],        # ₹950 per share
    "trade_value_crore": [190.0],        # ₹190 crore in total
})

verdict = knn.predict(scaler.transform(my_deal))[0]
print("The model's verdict:", verdict)

Try a tiny deal — say `quantity = 100000`, `price_inr = 50`, `trade_value_crore = 0.5` — and watch the verdict change to BULK.

---

## Recap — the recipe you just learned

Every classification project follows this same skeleton:

1. **Load** the data and **look** at it (Steps 2–3)
2. **Fix problems** — we balanced a 92/8 dataset to 50/50 (Step 4)
3. **Choose** features X and label y (Step 5)
4. **Split** 75% train / 25% test, and never let them mix (Step 6)
5. **Scale** the features (Step 7)
6. **Train** the model (Step 8)
7. **Judge honestly** — accuracy vs a dumb baseline, confusion matrix, precision & recall (Steps 9–10)
8. **Tune** the dials, like K (Step 11)

Swap `KNeighborsClassifier` for any other model and steps 1–7 stay *exactly the same*. That's why this recipe is worth memorising.

## Homework (optional, fun)

1. **Change the clues:** remove `"trade_value_crore"` from the `features` list in Step 5 and re-run everything (Runtime → Run all). Surprise — you lose almost nothing (~79.3%). Why? Because value = quantity × price, so the model can rebuild the missing clue from the other two. Now try keeping **only** `"trade_value_crore"`: one clue alone still scores ~71.3%. Clues can overlap — more features isn't always more information.
2. **Try to predict the impossible:** change the label in Step 5 to `y = data["side"]` (BUY vs SELL) and re-run. Watch accuracy crash to ~50% — a coin flip. Lesson: if the clues contain no real signal about the answer, no model can save you. Knowing *when* ML won't work is as valuable as knowing when it will.
